# 02 - Transformação

## Pipeline de Transformação e Engenharia de Atributos (02_transform)

### Visão Geral e Arquitetura do Pipeline

Este notebook implementa a camada de transformação (Silver) e a construção das Tabelas Analíticas de Modelagem (ABT - Gold). O objetivo é processar os dados brutos de crédito, consolidar a variável resposta e estruturar os dados para os experimentos de modelagem no notebook 03_modeling.

O pipeline opera inteiramente em memória para transformações intermediárias, persistindo em disco exclusivamente as bases consolidadas para modelagem.

---

### Fases do Pipeline

1. **Higienização e Tipagem Estrutural (Camada Silver):**
   - cadastral_tratada: relação 1:1 por cliente com consolidação por recência e fallback.
   - emprestimos_tratada: relação 1:N por contrato, com conservação volumétrica integral.
   - parcelas_tratada: relação 1:M por prestação, com remoção estrita de duplicatas exatas.

2. **Rollup de Parcelas e Construção do Target:**
   - Agrupamento em memória gerando o DataFrame parcelas_por_contrato (grão de contrato).
   - Filtragem pela versão mais recente da parcela e consolidação de pagamentos fracionados (tolerância de R$ 0,01).
   - Apuração do horizonte de observação MOB 12 via tempo corrido (data de decisão do contrato + 360 dias).
   - Critério de maturação de safra referenciado no limite empírico observável da carteira: max(data_real_pagamento).
   - Construção do target OVER60MOB12 (1 para atraso superior a 60 dias dentro do MOB 12; 0 para contratos maduros sem atraso crítico; NA para safras imaturas ou com incerteza).

3. **ABT de Baseline (Benchmark sem Engenharia):**
   - Merge estrutural direto das bases tratadas com a target.
   - Sem criação de novas features derivadas.
   - Sem encoding, escalonamento ou imputações manuais.
   - Persistência em disco: **dados/abt_baseline.parquet**.

4. **Engenharia de Atributos (Feature Engineering):**
   - Features comportamentais históricas calculadas nas bases individuais (rollups por cliente).
   - Merge analítico completo.
   - Features de interação cruzadas pós-merge (idade na decisão, comprometimento de renda, razões financeiras).

5. **ABT Final para Modelagem:**
   - Validações de integridade, ausência de leakage e tipagem final.
   - Persistência em disco: dados/abt_modelagem.parquet.

6. **Preparação da Base de Inferência (Submissão):**
   - Aplicação estrita dos mesmos tratamentos cadastrais da base de treino.
   - Consolidação das features comportamentais históricas para a data de solicitação.
   - Persistência da base de escoragem para predição em produção.

---

### Diretrizes de Engenharia e Boas Práticas

- **Prevenção de Vazamento Temporal (Data Leakage):** Apenas informações disponíveis no momento da tomada de decisão (data_decisao no histórico e data_solicitacao na submissão) podem ser utilizadas como preditores. Variáveis de desfecho do contrato não entram como features.
- **Validação por Invariantes:** Asserções de pipeline (assert) devem testar consistência lógica e conservação volumétrica dinâmica entre entradas e saídas, evitando valores fixos (magic numbers).
- **Critério de Maturação da Safra (Target):**
  - **Evento Mau (1):** Contratos com atraso superior a 60 dias até a 12ª parcela.
  - **Evento Bom (0):** Contratos maduros (pelo menos 12 parcelas observadas ou com mais de 365 dias de histórico) sem atraso superior a 60 dias.
  - **Indeterminado (NA):** Contratos imaturos sem atraso crítico (excluídos do treinamento supervisionado).

In [ ]:
import sys
sys.path.append("..")

import src.functions as fn #Funcoes criadas no projeto
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Carrega cópias de trabalho mantendo os arquivos originais intactos.
cadastral = pd.read_parquet("../dados/base_cadastral.parquet").copy()
submissao = pd.read_parquet("../dados/base_submissao.parquet").copy()
emprestimos = pd.read_parquet("../dados/historico_emprestimos.parquet").copy()
parcelas = pd.read_parquet("../dados/historico_parcelas.parquet").copy()

print(
    f"cadastral: {cadastral.shape} | submissao: {submissao.shape} | "
    f"emprestimos: {emprestimos.shape} | parcelas: {parcelas.shape}"
)

## Tratamento de Dados para as tabelas-base de treino

In [ ]:
# --- TRATAMENTO E CONSOLIDAÇÃO: BASE CADASTRAL ---

# 1. Parâmetros e referências dinâmicas (a base de partida não é modificada)
ano_limite_nascimento = 1920
formatos_data_cad = ["%Y-%m-%d", "%d/%m/%Y"]
colunas_inteiras_cad = ["qtd_membros_familia", "qtd_filhos"]
cols_para_remover = ["sexo"]
formato_data_br_regex = r"^\d{2}/\d{2}/\d{4}$"   # estrutura DD/MM/AAAA (data pode ser impossível)

n_clientes_esperados = cadastral["id_cliente"].nunique()
cadastral_proc = cadastral.copy()

# 2. Conversão de tipos (datas com formato explícito; renda só se vier como texto)
numericas_texto_cad = {}
if not pd.api.types.is_numeric_dtype(cadastral_proc["renda_anual"]):
    numericas_texto_cad["renda_anual"] = {"decimal": ",", "milhar": "."}

cadastral_proc, rel_tipos = fn.converter_tipos(
    cadastral_proc,
    datas={"data_nascimento": formatos_data_cad},
    numericas_texto=numericas_texto_cad or None,
    inteiros=colunas_inteiras_cad,
)

# 3. Explica cada falha de conversão, comparando com a base de origem
#    Data: falha esperada = texto no formato DD/MM/AAAA que não existe no calendário
txt_nasc = cadastral["data_nascimento"].astype("string")
falha_nasc = cadastral_proc["data_nascimento"].isna() & txt_nasc.notna()
nasc_explicada = falha_nasc & txt_nasc.str.match(formato_data_br_regex).fillna(False)

#    Renda: falha esperada = placeholder de ausência (padronizado como nulo na função)
placeholders_norm = {fn._normalizar(p) for p in fn.PLACEHOLDERS_PADRAO}
txt_renda = fn._normalizar_serie(cadastral["renda_anual"])
falha_renda = cadastral_proc["renda_anual"].isna() & cadastral["renda_anual"].notna()
renda_explicada = falha_renda & txt_renda.isin(placeholders_norm)

n_nasc_inexplicadas = int((falha_nasc & ~nasc_explicada).sum())
n_renda_inexplicadas = int((falha_renda & ~renda_explicada).sum())

print(f"data_nascimento: {int(falha_nasc.sum())} falhas | {int(nasc_explicada.sum())} datas impossíveis (DD/MM/AAAA)")
print(f"renda_anual: {int(falha_renda.sum())} falhas | {int(renda_explicada.sum())} placeholders")

assert n_nasc_inexplicadas == 0, (
    f"{n_nasc_inexplicadas} datas falharam fora do padrão DD/MM/AAAA: "
    f"{txt_nasc[falha_nasc & ~nasc_explicada].unique()[:5]}"
)
assert n_renda_inexplicadas == 0, (
    f"{n_renda_inexplicadas} rendas falharam sem ser placeholder: "
    f"{cadastral['renda_anual'][falha_renda & ~renda_explicada].unique()[:5]}"
)

# 4. Data de nascimento sentinela: flag a partir da data já convertida, valor vira nulo
ano_nascimento = cadastral_proc["data_nascimento"].dt.year
cadastral_proc["data_nascimento_invalida"] = ano_nascimento.le(ano_limite_nascimento)
cadastral_proc["data_nascimento"] = cadastral_proc["data_nascimento"].mask(
    cadastral_proc["data_nascimento_invalida"]
)

# 5. Padronização de categóricas e placeholders -> nulo
cols_categoricas = list(
    cadastral_proc.select_dtypes(include=["object", "string", "category"]).columns
)
cadastral_proc, rel_cat = fn.padronizar_categoricas(cadastral_proc, colunas=cols_categoricas)

# 6. Consolidação de duplicatas por id_cliente
#    Premissa documentada: a ordem das linhas indica recência (a última ocorrência é a mais recente).
#    last() devolve o último valor NÃO nulo de cada coluna.
n_linhas_antes_dedup = len(cadastral_proc)
max_ocorrencias = int(cadastral_proc["id_cliente"].value_counts().max())

cadastral_tratada = cadastral_proc.groupby("id_cliente", sort=True, as_index=False).last()

# A flag só vale se o valor consolidado continuar nulo
cadastral_tratada["data_nascimento_invalida"] = (
    cadastral_tratada["data_nascimento_invalida"].astype("bool")
    & cadastral_tratada["data_nascimento"].isna()
)

# 7. Tipos finais e remoção de colunas fora do modelo
cadastral_tratada["id_cliente"] = cadastral_tratada["id_cliente"].astype("int64")
for col in cols_categoricas:
    cadastral_tratada[col] = cadastral_tratada[col].astype("category")

n_cols_antes_drop = cadastral_tratada.shape[1]
cadastral_tratada = cadastral_tratada.drop(columns=cols_para_remover)

# 8. Asserções de integridade
assert len(cadastral_tratada) == n_clientes_esperados, (
    f"Esperado {n_clientes_esperados} clientes, obtido {len(cadastral_tratada)}"
)
assert cadastral_tratada["id_cliente"].is_unique, "id_cliente não é único na base consolidada"
assert cadastral_tratada.shape[1] == n_cols_antes_drop - len(cols_para_remover), (
    "Inconsistência no número de colunas após exclusão"
)
assert not (
    cadastral_tratada["data_nascimento_invalida"] & cadastral_tratada["data_nascimento"].notna()
).any(), "Flag de data inválida contradiz data_nascimento preenchida"
assert pd.api.types.is_datetime64_any_dtype(cadastral_tratada["data_nascimento"]), (
    "data_nascimento não é datetime64"
)

# 9. Relatório
print("--- Base Cadastral Tratada ---")
print(f"Shape final: {cadastral_tratada.shape}")
print(f"Linhas antes da deduplicação: {n_linhas_antes_dedup} | máximo de ocorrências por cliente: {max_ocorrencias}")
print(f"Clientes com data_nascimento_invalida: {int(cadastral_tratada['data_nascimento_invalida'].sum())}")
display(rel_tipos)
display(rel_cat)

In [ ]:
cadastral_tratada.info()

In [ ]:
cadastral_tratada.head()

In [ ]:
import numpy as np

# --- TRATAMENTO E TIPAGEM: BASE EMPRESTIMOS ---

# 1. Schema declarado (37 colunas após a renomeação, conforme .info())
formatos_data_emp = ["%Y-%m-%d", "%d/%m/%Y"]
mapa_renomeacao_emp = {
    "valor_credito": "valor_credito_concedido",  # valor efetivamente concedido (pós-decisão)
    "valor_solicitado": "valor_credito",         # alinha com a submissão
}
cols_descartar_emp = ["dia_semana_solicitacao", "hora_solicitacao"]  # mesma decisão da submissão
cols_id_emp = ["id_contrato", "id_cliente"]
cols_datas_emp = [
    "data_decisao", "data_liberacao", "data_primeiro_vencimento",
    "data_ultimo_vencimento_original", "data_ultimo_vencimento", "data_encerramento",
]
cols_float_emp = [
    "valor_credito", "valor_credito_concedido", "valor_bem", "valor_parcela",
    "valor_entrada", "percentual_entrada", "taxa_juros_padrao", "taxa_juros_promocional",
]
cols_inteiras_nulaveis_emp = ["qtd_parcelas_planejadas", "indicador_seguro_contratado"]
cols_inteiras_emp = ["area_venda", "indicador_ultima_solicitacao_dia"]

# Indicador textual (sim/nao) convertido para 1/0 após a padronização
col_indicador_texto = "indicador_ultima_solicitacao_contrato"
valor_positivo_ind, valor_negativo_ind = "sim", "nao"

cols_cat_emp = [
    "tipo_contrato", "status_contrato", "tipo_pagamento", "finalidade_emprestimo",
    "tipo_cliente", "faixa_rendimento", "tipo_portfolio", "tipo_produto",
    "categoria_bem", "combinacao_produto", "setor_vendedor", "canal_venda",
    "motivo_recusa", "acompanhantes_cliente",
]
schema_esperado_emp = (
    cols_id_emp + cols_datas_emp + cols_float_emp + cols_inteiras_nulaveis_emp
    + cols_inteiras_emp + cols_cat_emp + [col_indicador_texto] + cols_descartar_emp
)

n_linhas_esperadas_emp = len(emprestimos)
emprestimos_proc = emprestimos.copy()  # a base de partida não é modificada

# 2. Renomeação, com checagem de que as colunas de origem existem
ausentes_renomeacao = set(mapa_renomeacao_emp) - set(emprestimos_proc.columns)
assert not ausentes_renomeacao, f"Colunas ausentes para renomear: {ausentes_renomeacao}"
emprestimos_proc = emprestimos_proc.rename(columns=mapa_renomeacao_emp)

# 3. Schema: nenhuma coluna a mais nem a menos que o declarado
assert len(schema_esperado_emp) == len(set(schema_esperado_emp)), "Coluna declarada em mais de um grupo"
assert set(emprestimos_proc.columns) == set(schema_esperado_emp), (
    f"Colunas fora do schema: {set(emprestimos_proc.columns) - set(schema_esperado_emp)} | "
    f"ausentes: {set(schema_esperado_emp) - set(emprestimos_proc.columns)}"
)

# 4. Descarte das colunas definidas (antes das conversões, para não processá-las à toa)
emprestimos_proc = emprestimos_proc.drop(columns=cols_descartar_emp)

# 5. Conversão de datas (formato explícito) e inteiros com nulos
emprestimos_proc, rel_tipos_emp = fn.converter_tipos(
    emprestimos_proc,
    datas={c: formatos_data_emp for c in cols_datas_emp},
    inteiros=["qtd_parcelas_planejadas"],
)
assert (rel_tipos_emp["falhas"] == 0).all(), (
    f"Conversão descartou valores:\n{rel_tipos_emp[rel_tipos_emp['falhas'] > 0]}"
)

# 6. Padronização das categóricas e do indicador textual (minúsculas, sem acento, placeholders -> nulo)
cols_padronizar_emp = cols_cat_emp + [col_indicador_texto]
emprestimos_proc, rel_cat_emp = fn.padronizar_categoricas(emprestimos_proc, colunas=cols_padronizar_emp)

# 7. Indicador sim/nao -> 1/0 (vetorizado; para se houver valor fora do esperado ou nulo)
ind_padronizado = emprestimos_proc[col_indicador_texto]
valores_validos_ind = {valor_positivo_ind, valor_negativo_ind}
fora_do_esperado = ind_padronizado.isna() | ~ind_padronizado.isin(valores_validos_ind)
assert not fora_do_esperado.any(), (
    f"{col_indicador_texto}: {int(fora_do_esperado.sum())} valores fora de {valores_validos_ind} "
    f"(exemplos: {ind_padronizado[fora_do_esperado].unique()[:5]})"
)
emprestimos_proc[col_indicador_texto] = np.where(
    ind_padronizado.eq(valor_positivo_ind), 1, 0
).astype("int8")

# 8. Tipos finais
emprestimos_proc[cols_id_emp] = emprestimos_proc[cols_id_emp].astype("int64")
emprestimos_proc[cols_float_emp] = emprestimos_proc[cols_float_emp].astype("float64")
emprestimos_proc[cols_inteiras_emp] = emprestimos_proc[cols_inteiras_emp].astype("int64")
emprestimos_proc["indicador_seguro_contratado"] = emprestimos_proc["indicador_seguro_contratado"].astype("Int8")
emprestimos_proc[col_indicador_texto] = emprestimos_proc[col_indicador_texto].astype("Int8")
emprestimos_proc[cols_cat_emp] = emprestimos_proc[cols_cat_emp].astype("category")

emprestimos_tratada = emprestimos_proc

# 9. Asserções de integridade
assert len(emprestimos_tratada) == n_linhas_esperadas_emp, (
    f"Perda de registros: esperado {n_linhas_esperadas_emp}, obtido {len(emprestimos_tratada)}"
)
assert emprestimos_tratada["id_contrato"].is_unique, "id_contrato não é único"
assert emprestimos_tratada["data_decisao"].notna().all(), "data_decisao possui nulos (origem da target)"
assert not set(cols_descartar_emp) & set(emprestimos_tratada.columns), "Colunas descartadas ainda presentes"
assert emprestimos_tratada.shape[1] == len(schema_esperado_emp) - len(cols_descartar_emp), (
    "Número de colunas inconsistente após o descarte"
)
for col in cols_datas_emp:
    assert pd.api.types.is_datetime64_any_dtype(emprestimos_tratada[col]), (
        f"{col} deveria ser datetime64, está como {emprestimos_tratada[col].dtype}"
    )

# Conversão do indicador preserva a contagem de cada categoria original
n_sim_original = int(ind_padronizado.eq(valor_positivo_ind).sum())
assert int(emprestimos_tratada[col_indicador_texto].sum()) == n_sim_original, (
    "Contagem de 1 no indicador difere da contagem de 'sim' antes da conversão"
)
assert set(emprestimos_tratada[col_indicador_texto].unique()) <= {0, 1}, "Indicador com valor fora de {0, 1}"

# 10. Coerência de tipo_contrato com a submissão (invariante dinâmica)
cats_emp = set(emprestimos_tratada["tipo_contrato"].dropna().astype(str).unique())
cats_sub = set(submissao_tratada["tipo_contrato"].dropna().astype(str).unique())
assert cats_emp == cats_sub, (
    f"tipo_contrato diverge | só em empréstimos: {cats_emp - cats_sub} | só em submissão: {cats_sub - cats_emp}"
)

# 11. Relatório
print("--- Base Empréstimos Tratada ---")
print(f"Shape final: {emprestimos_tratada.shape}")
print(f"Contratos únicos: {emprestimos_tratada['id_contrato'].nunique()} | Clientes distintos: {emprestimos_tratada['id_cliente'].nunique()}")
display(rel_tipos_emp)
display(rel_cat_emp)

In [ ]:
emprestimos_tratada.info()

In [ ]:
emprestimos_tratada.head()

In [ ]:
# --- TRATAMENTO E TIPAGEM: HISTÓRICO DE PARCELAS ---

# 1. Schema declarado
formatos_data_parc = ["%Y-%m-%d", "%d/%m/%Y"]
cols_id_parc = ["id_contrato", "id_cliente"]
cols_inteiras_parc = ["versao_parcela", "numero_parcela"]
cols_datas_parc = ["data_prevista_pagamento", "data_real_pagamento"]
cols_float_parc = ["valor_previsto_parcela", "valor_pago_parcela"]
schema_esperado_parc = cols_id_parc + cols_inteiras_parc + cols_datas_parc + cols_float_parc

assert len(schema_esperado_parc) == len(set(schema_esperado_parc)), "Coluna declarada em mais de um grupo"
assert set(parcelas.columns) == set(schema_esperado_parc), (
    f"Colunas fora do schema: {set(parcelas.columns) - set(schema_esperado_parc)} | "
    f"ausentes: {set(schema_esperado_parc) - set(parcelas.columns)}"
)

# 2. Duplicatas integrais (cópia de trabalho; a base de partida não é modificada)
n_linhas_originais_parc = len(parcelas)
n_duplicadas_integrais_parc = int(parcelas.duplicated().sum())
parcelas_proc = parcelas.drop_duplicates().copy()

# 3. Conversão de datas (formato explícito) e inteiros com nulos
parcelas_proc, rel_tipos_parc = fn.converter_tipos(
    parcelas_proc,
    datas={c: formatos_data_parc for c in cols_datas_parc},
    inteiros=cols_inteiras_parc,
)

# 4. Tipos finais (astype("int64") nos ids falha se houver nulos)
parcelas_proc[cols_id_parc] = parcelas_proc[cols_id_parc].astype("int64")
parcelas_proc[cols_float_parc] = parcelas_proc[cols_float_parc].astype("float64")

parcelas_tratada = parcelas_proc

# 5. Asserções de integridade estrutural
assert len(parcelas_tratada) == n_linhas_originais_parc - n_duplicadas_integrais_parc, (
    "Inconsistência volumétrica após remoção de duplicatas"
)
assert not parcelas_tratada.duplicated().any(), "Ainda existem duplicatas integrais"
assert set(parcelas_tratada.columns) == set(schema_esperado_parc), "Schema alterado durante o tratamento"
for col in cols_datas_parc:
    assert pd.api.types.is_datetime64_any_dtype(parcelas_tratada[col]), f"{col} não é datetime64"

# 6. Relatório (falhas de conversão exibidas para análise, não bloqueadas)
print("--- Base Parcelas Tratada ---")
print(f"Linhas originais: {n_linhas_originais_parc} | duplicatas integrais removidas: {n_duplicadas_integrais_parc}")
print(f"Shape final: {parcelas_tratada.shape}")
print(f"Contratos distintos: {parcelas_tratada['id_contrato'].nunique()} | Clientes distintos: {parcelas_tratada['id_cliente'].nunique()}")
print(f"Total de falhas de conversão: {int(rel_tipos_parc['falhas'].sum())}")
display(rel_tipos_parc)
print(parcelas_tratada.dtypes)

# 7. Diagnóstico de coerência (somente leitura, nada é alterado)
chave_parc = ["id_contrato", "numero_parcela", "versao_parcela"]
print("\nNulos por coluna:")
print(parcelas_tratada.isna().sum())
print(f"\nLinhas com chave {chave_parc} repetida: {int(parcelas_tratada.duplicated(subset=chave_parc, keep=False).sum())}")

contratos_parc = set(parcelas_tratada["id_contrato"].unique())
contratos_emp = set(emprestimos_tratada["id_contrato"].unique())
print(f"Contratos em parcelas sem registro em empréstimos: {len(contratos_parc - contratos_emp)}")
print(f"Contratos em empréstimos sem nenhuma parcela: {len(contratos_emp - contratos_parc)}")

# O id_cliente de cada parcela deve coincidir com o do contrato em empréstimos
cliente_por_contrato = emprestimos_tratada.set_index("id_contrato")["id_cliente"]
cliente_esperado = parcelas_tratada["id_contrato"].map(cliente_por_contrato)
divergentes = cliente_esperado.notna() & (cliente_esperado != parcelas_tratada["id_cliente"])
print(f"Parcelas com id_cliente diferente do contrato em empréstimos: {int(divergentes.sum())}")

In [ ]:
parcelas_tratada.info()

In [ ]:
parcelas_tratada.head()

## Documentação de Decisões: Higienização e Tipagem das Bases Históricas

Esta etapa estabelece a camada de dados limpos (*Silver*) para o desenvolvimento do modelo, padronizando formatos, resolvendo inconsistências estruturais e garantindo integridade relacional sem vazamento de dados. A base de inferência (`submissao`) é preservada para avaliação posterior.

---

### 1. Base `cadastral` (`cadastral_tratada`)

- **Datas Sentinela e Inconsistentes:**
  - Identificação de anos de nascimento irrealistas (ano <= 1920).
  - Criação da flag booleana `data_nascimento_invalida` para rastreabilidade de negócio.
  - Conversão dos valores anômalos em `data_nascimento` para nulo (`pd.NaT`).
- **Conversão de Tipos Estruturais:**
  - `data_nascimento`: convertida para `datetime64[ns]` via múltiplos formatos (`%Y-%m-%d`, `%d/%m/%Y`).
  - `renda_anual`: higienização textual de moedas e conversão para `float64`.
  - `qtd_membros_familia`: convertida para inteiro com suporte a nulos (`Int64`).
- **Padronização Categórica e Placeholders:**
  - Remoção de acentuação, padronização em minúsculas e eliminação de espaços duplicados em todas as variáveis textuais.
  - Conversão de placeholders de ausência disfarçada (ex.: `"não informado"`, `"unknown"`, `"-"`) em `NaN` via `fn.padronizar_categoricas`.
- **Deduplicação e Atualização Cadastral:**
  - Clientes repetidos consolidados via `.combine_first()`.
  - A 2ª ocorrência (registro mais recente) prevalece como fonte primária; valores ausentes herdam os dados preenchidos na 1ª ocorrência.
- **Otimização de Memória e Conformidade:**
  - Conversão de `id_cliente` para `int64`, `qtd_filhos` para `Int64` e colunas textuais para `category`.
  - Remoção da coluna sensível `sexo` da base de modelagem para assegurar equidade algorítmica (*fairness*).
- **Validação de Integridade (Asserções Dinâmicas):**
  - Relação 1:1 estrita: total de linhas final igual ao número de `id_cliente` únicos originais (50.000).

---

### 2. Base `emprestimos` (`emprestimos_tratada`)

- **Preservação Volumétrica e Integridade Histórica:**
  - Manutenção de todos os contratos e histórico original (211.944 linhas), sem exclusão de linhas nem engenharia antecipada.
- **Conversão Temporal:**
  - `data_decisao` e `data_primeiro_vencimento`: convertidas para `datetime64[ns]`.
- **Padronização Categórica:**
  - Higienização textual e conversão de placeholders em `NaN` em atributos como `tipo_contrato`, `status_contrato`, etc.
- **Otimização de Memória e Tipagem:**
  - `id_contrato` e `id_cliente`: tipificados como `int64`.
  - Colunas categóricas textuais convertidas para `category`.
- **Validação de Integridade (Asserções Dinâmicas):**
  - Conservação exata da volumetria de entrada (`len(emprestimos_tratada) == len(emprestimos)`).
  - Unicidade estrita da chave primária `id_contrato` comprovada via assert.

---

### 3. Base `parcelas` (`parcelas_tratada`)

- **Eliminação de Duplicatas Técnicas:**
  - Remoção estrita de duplicatas integrais exatas (`drop_duplicates()`), preservando pagamentos parciais e amortizações legítimas.
- **Conversão Temporal:**
  - `data_prevista_pagamento` e `data_pagamento`: padronizadas para `datetime64[ns]`.
- **Otimização Estrutural e Tipagem:**
  - `id_contrato`: tipificado como `int64`.
  - `versao_parcela` e `numero_parcela`: convertidos para inteiros anuláveis (`Int64`).
- **Validação de Integridade (Asserções Dinâmicas):**
  - Conservação volumétrica exata calculada dinamicamente: `n_linhas_originais - n_duplicadas_integrais`.
  - Assert garantindo zero duplicatas integrais remanescentes na saída.

---

### Resumo Comparativo de Contratos de Saída

| Base | Chave Primária | Relação | Linhas Originais | Linhas Finais | Ação Principal de Limpeza |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **`cadastral_tratada`** | `id_cliente` | 1 : 1 | 50.600 | 50.000 | Consolidação por recência + fallback; exclusão de `sexo`; flag de nascimento. |
| **`emprestimos_tratada`** | `id_contrato` | 1 : N (cliente) | 211.944 | 211.944 | Conversão de datas e categorias; conservação estrita de contratos. |
| **`parcelas_tratada`** | Composta (`id_contrato`, `versao`, `num`) | 1 : M (contrato) | 1.664.384 | 1.659.408 | Exclusão de 4.976 duplicatas integrais; tipagem temporal e de contadores. |

## Rollup de parcelas por contrato e criação da target

### OVER60MOB12

In [ ]:
# --- ROLLUP DE PARCELAS POR CONTRATO E TARGET OVER60MOB12 (VARIANTE "FORA") ---

# 1. Parâmetros e regras de negócio documentadas
tolerancia_centavos = 0.01
limite_dias_atraso = 60
dias_mes_mob = 30
multiplicador_mob = 12
dias_janela_mob = dias_mes_mob * multiplicador_mob  # 360 dias
chave_parcela = ["id_contrato", "numero_parcela"]
nome_target = f"OVER{limite_dias_atraso}MOB{multiplicador_mob}"

# Referências estruturais e data de corte factual
n_contratos_esperados_parc = parcelas_tratada["id_contrato"].nunique()
data_corte_sistema = parcelas_tratada["data_real_pagamento"].max()

assert pd.api.types.is_datetime64_any_dtype(emprestimos_tratada["data_decisao"]), (
    "data_decisao em emprestimos_tratada não está em datetime64"
)

# 2. Versão ativa mais recente de cada prestação
max_versoes = parcelas_tratada.groupby(chave_parcela)["versao_parcela"].transform("max")
parc_ativas = parcelas_tratada.loc[parcelas_tratada["versao_parcela"] == max_versoes].copy()

# Pagamento positivo sem data (incerteza), marcado por linha antes de agregar
parc_ativas["_pgto_sem_data"] = (
    parc_ativas["valor_pago_parcela"].gt(0) & parc_ativas["data_real_pagamento"].isna()
)

# 3. Consolidação no grão (id_contrato, numero_parcela)
resumo_parcelas = (
    parc_ativas.groupby(chave_parcela)
    .agg(
        data_prevista_pagamento=("data_prevista_pagamento", "first"),
        valor_previsto_parcela=("valor_previsto_parcela", "first"),
        total_pago=("valor_pago_parcela", "sum"),
        ultima_data_pagamento=("data_real_pagamento", "max"),
        pagamento_sem_data=("_pgto_sem_data", "any"),
    )
    .reset_index()
)

# Quitação (acumulado cobre o previsto, com tolerância) e flag de incerteza
resumo_parcelas["quitada"] = (
    resumo_parcelas["total_pago"]
    >= (resumo_parcelas["valor_previsto_parcela"] - tolerancia_centavos)
)
incerto = (
    resumo_parcelas["pagamento_sem_data"]
    | ~resumo_parcelas["valor_previsto_parcela"].gt(0)
    | resumo_parcelas["data_prevista_pagamento"].isna()
)

# 4. Data de liquidação: último pagamento se quitada; vencimento se quitada sem data; corte se em aberto
liquidacao_quitada = resumo_parcelas["ultima_data_pagamento"].fillna(
    resumo_parcelas["data_prevista_pagamento"]
)
resumo_parcelas["data_liquidacao"] = liquidacao_quitada.where(
    resumo_parcelas["quitada"], data_corte_sistema
)

dias_diferenca = (
    resumo_parcelas["data_liquidacao"] - resumo_parcelas["data_prevista_pagamento"]
).dt.days
resumo_parcelas["dias_atraso"] = dias_diferenca.clip(lower=0).astype("Int64").mask(incerto)

# 5. Origem fixa em data_decisao (sem fallback e sem persistir na tabela final)
origem_contrato = (
    emprestimos_tratada[["id_contrato", "data_decisao"]]
    .drop_duplicates("id_contrato")
    .rename(columns={"data_decisao": "_data_origem"})
)
base_mob = resumo_parcelas.merge(origem_contrato, on="id_contrato", how="left")
assert base_mob["_data_origem"].notna().all(), "Há parcelas de contratos sem data_decisao"

# 6. Avaliação temporal da janela MOB 12
fim_janela_prestacao = base_mob["_data_origem"] + pd.Timedelta(days=dias_janela_mob)
limiar_na_janela = (
    base_mob["data_prevista_pagamento"] + pd.Timedelta(days=limite_dias_atraso)
) <= fim_janela_prestacao

evento_no_mob = base_mob["dias_atraso"].ge(limite_dias_atraso) & limiar_na_janela
incerta_no_mob = base_mob["dias_atraso"].isna() & limiar_na_janela

# 7. Agregação no grão de contrato
contratos_unicos = (
    base_mob[["id_contrato", "_data_origem"]]
    .drop_duplicates("id_contrato")
    .set_index("id_contrato")
)
teve_evento = evento_no_mob.groupby(base_mob["id_contrato"]).any().reindex(contratos_unicos.index)
teve_incerteza = incerta_no_mob.groupby(base_mob["id_contrato"]).any().reindex(contratos_unicos.index)

janela_maturada = (
    contratos_unicos["_data_origem"] + pd.Timedelta(days=dias_janela_mob)
) <= data_corte_sistema

# Métricas comportamentais no horizonte MOB 12 (diagnóstico, não usar como feature)
prestacoes_no_mob = base_mob.loc[base_mob["data_prevista_pagamento"] <= fim_janela_prestacao]
max_dpd_mob12 = prestacoes_no_mob.groupby("id_contrato")["dias_atraso"].max()
qtd_prestacoes_mob12 = prestacoes_no_mob.groupby("id_contrato")["numero_parcela"].count()

# 8. Target, variante "fora": 0 = maduro sem incerteza; 1 = maduro com evento; <NA> = demais
target_serie = pd.Series(pd.NA, index=contratos_unicos.index, dtype="Int8")
target_serie = target_serie.mask(janela_maturada & ~teve_incerteza, 0)
target_serie = target_serie.mask(janela_maturada & teve_evento, 1)

parcelas_por_contrato = pd.DataFrame(index=contratos_unicos.index)
parcelas_por_contrato["max_dias_atraso_mob12"] = max_dpd_mob12.reindex(contratos_unicos.index)
parcelas_por_contrato["qtd_parcelas_mob12"] = (
    qtd_prestacoes_mob12.reindex(contratos_unicos.index).fillna(0).astype("Int64")
)
parcelas_por_contrato["contrato_maduro_mob12"] = janela_maturada
parcelas_por_contrato[nome_target] = target_serie
parcelas_por_contrato = parcelas_por_contrato.reset_index()

# 9. Asserções de integridade
assert len(parcelas_por_contrato) == n_contratos_esperados_parc, (
    f"Inconsistência volumétrica: esperado {n_contratos_esperados_parc}, obtido {len(parcelas_por_contrato)}"
)
assert parcelas_por_contrato["id_contrato"].is_unique, "Falha de unicidade em id_contrato"
assert "_data_origem" not in parcelas_por_contrato.columns, "_data_origem não deve ser persistida"

# 10. Relatório de validação
n_bons = int(target_serie.eq(0).sum())
n_maus = int(target_serie.eq(1).sum())
n_indet = int(target_serie.isna().sum())
taxa_inadimplencia = n_maus / (n_bons + n_maus) * 100

print("--- Rollup de Parcelas por Contrato Concluído ---")
print(f"Data de corte: {data_corte_sistema:%Y-%m-%d}")
print(f"Total de contratos consolidados: {len(parcelas_por_contrato)}")
print(f"Bons (0): {n_bons} | Maus (1): {n_maus} | Indeterminados (<NA>): {n_indet}")
print(f"Taxa de inadimplência observada: {taxa_inadimplencia:.4f}%")

## Tratamento - base submissao


In [ ]:
# --- TRATAMENTO E TIPAGEM: SUBMISSAO ---

# 1. Referências para validação por invariantes
n_linhas_originais_sub = len(submissao)
marcador_data_invalida = "2025-13-01"
formatos_data_sub = ["%Y-%m-%d", "%d/%m/%Y"]
colunas_numericas_sub = ["valor_credito", "valor_bem", "valor_parcela"]
colunas_descartadas_sub = ["hora_solicitacao", "dia_semana_solicitacao"]

submissao_tratada = submissao.copy()

# 2. Flag da data inválida, criada antes da conversão (a string fixa viraria nulo)
submissao_tratada["data_invalida"] = (
    submissao_tratada["data_solicitacao"].astype("string").eq(marcador_data_invalida)
)

# 3. Padronização de tipo_contrato (a função retorna DataFrame e relatório)
submissao_tratada, relatorio_padronizacao_sub = fn.padronizar_categoricas(
    submissao_tratada, ["tipo_contrato"]
)

# 4. Conversão das datas: ISO primeiro, DD/MM/AAAA depois; o marcador inválido vira nulo
submissao_tratada, relatorio_tipos_sub = fn.converter_tipos(
    submissao_tratada,
    datas={"data_solicitacao": formatos_data_sub},
)

# 5. Remoção de hora e dia da semana da solicitação (a base bruta permanece intacta)
submissao_tratada = submissao_tratada.drop(columns=colunas_descartadas_sub)

# 6. Asserções de integridade
n_nulos_data = int(submissao_tratada["data_solicitacao"].isna().sum())
n_invalidas = int(submissao_tratada["data_invalida"].sum())

assert len(submissao_tratada) == n_linhas_originais_sub, "Número de linhas alterado"
assert n_nulos_data == n_invalidas, (
    f"Datas nulas ({n_nulos_data}) diferem das inválidas sinalizadas ({n_invalidas}): há formato não previsto"
)
assert pd.api.types.is_datetime64_any_dtype(submissao_tratada["data_solicitacao"]), (
    "data_solicitacao não foi convertida para datetime64"
)
assert not set(colunas_descartadas_sub) & set(submissao_tratada.columns), (
    "Colunas descartadas ainda presentes em submissao_tratada"
)
assert set(colunas_descartadas_sub) <= set(submissao.columns), (
    "A base bruta submissao deveria manter as colunas descartadas"
)
for coluna in colunas_numericas_sub:
    assert submissao_tratada[coluna].equals(submissao[coluna]), f"{coluna} foi alterada"

# 7. Relatórios
print("--- Base Submissao Tratada ---")
print(f"Linhas: {len(submissao_tratada):,}")
print(f"Colunas descartadas: {colunas_descartadas_sub}")
print(f"Datas inválidas -> nulo: {n_invalidas} ({n_invalidas / len(submissao_tratada):.2%})")
display(relatorio_padronizacao_sub)
display(relatorio_tipos_sub)
print(submissao_tratada["tipo_contrato"].value_counts(dropna=False))
print(submissao_tratada.dtypes)